# Stage 1 Validation — Causal Feature Pipeline (`F`) & Feast Offline Store

Interactive validation notebook for Tickets 01–02 inside the containerized Docker stack (`make up` / `make notebook`) or offline against a temporary directory.

In [ ]:
import os
import tempfile
from datetime import UTC, datetime
from pathlib import Path

from banking_fti.adapters import FeastFeatureStore
from banking_fti.core.contracts import RawBankingBatch
from banking_fti.core.validation import SchemaValidationError, validate_raw_batch
from banking_fti.features.causal_aggs import compute_card_window_aggs
from banking_fti.features.pipeline import FeaturePipeline
from banking_fti.features.simulator import BankingEventSimulator, SimulatorConfig

repo_path = Path(os.environ.get("FEAST_REPO_PATH", "feature_repo")).resolve()
default_data_dir = Path(tempfile.gettempdir()) / "banking_fti_feast_notebook"
data_dir = Path(os.environ.get("FEAST_DATA_DIR", str(default_data_dir))).resolve()

store = FeastFeatureStore(repo_path=repo_path, data_dir=data_dir)
registered_entities = sorted(entity.name for entity in store.list_entities())
registered_views = sorted(view.name for view in store.list_feature_views())
print("Entities:", registered_entities)
print("FeatureViews:", registered_views)

In [ ]:
sim = BankingEventSimulator(SimulatorConfig(num_customers=30, backfill_days=30, seed=42))
t0 = datetime(2026, 2, 1, 0, 0, tzinfo=UTC)
backfill = sim.generate_backfill(end_time=t0)

bad_apps = backfill.credit_applications.copy()
bad_apps.loc[0, "dti_ratio"] = -0.25
bad_batch = RawBankingBatch(
    customer_profiles=backfill.customer_profiles,
    card_events=backfill.card_events,
    credit_applications=bad_apps,
    batch_id="invalid-demo",
)

pandera_rejected_invalid = False
try:
    validate_raw_batch(bad_batch)
except SchemaValidationError as exc:
    pandera_rejected_invalid = True
    print("Pandera rejected invalid batch as expected:", exc.dataset_name)

assert pandera_rejected_invalid

In [ ]:
baseline_aggs = compute_card_window_aggs(backfill.card_events)
ordered = backfill.card_events.sort_values("event_timestamp").reset_index(drop=True)
target_cc = str(ordered.loc[0, "cc_num"])
future_indices = ordered.index[ordered["cc_num"] == target_cc]
mutated = ordered.copy()
if len(future_indices) > 1:
    mutated.loc[future_indices[-1], "amount"] = 99999.0

baseline_ordered = compute_card_window_aggs(ordered)
mutated_aggs = compute_card_window_aggs(mutated)
first_idx = int(future_indices[0])
causal_exact_match = baseline_ordered.iloc[first_idx].equals(mutated_aggs.iloc[first_idx])
print("Causal t - 1 invariance exact match:", causal_exact_match)
assert causal_exact_match

In [ ]:
pipeline = FeaturePipeline(feature_store=store)
backfill_res = pipeline.run_batch(backfill)
nominal_inc = sim.generate_incremental_batch(
    start_time=t0, window_hours=24, simulate_drift=False, batch_seq=1
)
nominal_res = pipeline.run_batch(
    nominal_inc,
    history_card_events=backfill.card_events,
    reference_stats=backfill_res.group_stats,
)
drift_inc = sim.generate_incremental_batch(
    start_time=t0, window_hours=24, simulate_drift=True, batch_seq=2
)
drift_res = pipeline.run_batch(
    drift_inc,
    history_card_events=backfill.card_events,
    reference_stats=backfill_res.group_stats,
)
drift_detected_on_inc = bool(drift_res.drift_detected)
print("Nominal drift_detected:", nominal_res.drift_detected)
print("Simulated drift_detected:", drift_detected_on_inc)
print("Parquet files:", sorted(p.name for p in store.data_dir.glob("*.parquet")))

In [ ]:
apps = store.read_feature_group("credit_applications")
entity_df = apps.loc[:, ["application_id", "customer_id", "event_timestamp"]].tail(10)

train_df = store.get_historical_features(
    entity_df,
    ["customer_profiles", "card_window_aggs", "credit_applications"],
)
batch_df = store.get_historical_features(
    entity_df,
    ["customer_profiles", "card_window_aggs", "credit_applications"],
    exclude_labels=True,
)

train_has_label = (
    "default_label" in train_df.columns and not bool(train_df["home_ownership"].isna().any())
)
batch_omits_label = "default_label" not in batch_df.columns
print("train_df shape:", train_df.shape, "has_label:", train_has_label)
print("batch_df shape:", batch_df.shape, "omits_label:", batch_omits_label)
train_df.head()